In [3]:
%pip -q install "qiskit==2.2.3" "numpy==2.3.5" "scipy==1.16.3"

In [6]:
import numpy as np
due = [1,2,3]
late_rate = [8,5,2]
setup = np.array([[0,2,7],[4,0,2],[3,6,0]])
cost = [max(0,t+1-due[j])*late_rate[j] for j in range(3) for t in range(3)]
pairs = {}
for t in range(2):
    for j in range(3):
        for k in range(3):
            if j != k:
                key = tuple(sorted((3*j+t,3*k+t+1)))
                pairs[key] = pairs.get(key,0)+float(setup[j,k])
print('지연 비용 계수:',cost)
print('전환 비용 이차계수:',pairs)

지연 비용 계수: [0, 8, 16, 0, 0, 5, 0, 0, 0]
전환 비용 이차계수: {(0, 4): 2.0, (0, 7): 7.0, (1, 3): 4.0, (3, 7): 2.0, (1, 6): 3.0, (4, 6): 6.0, (1, 5): 2.0, (1, 8): 7.0, (2, 4): 4.0, (4, 8): 2.0, (2, 7): 3.0, (5, 7): 6.0}


In [7]:
import time
import importlib.metadata as metadata
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

print('versions:', {k: metadata.version(k) for k in ['qiskit','numpy','scipy']})

def solve(names, cost, pairs, equalities, forbidden, feasible, describe,
          penalty=100.0, p=2, maxiter=160, shots=4096):
    n = len(names)
    a = np.array(cost, dtype=float)
    b = dict(pairs)
    constant = 0.0
    for indices, target in equalities:
        constant += penalty * target**2
        for i in indices:
            a[i] += penalty * (1 - 2 * target)
        for k, i in enumerate(indices):
            for j in indices[k+1:]:
                key = tuple(sorted((i,j)))
                b[key] = b.get(key,0.0) + 2*penalty
    for i,j in forbidden:
        key = tuple(sorted((i,j)))
        b[key] = b.get(key,0.0) + penalty

    # x_i=(1-Z_i)/2 치환을 적용한 Ising 계수
    h = -a/2
    offset = constant + a.sum()/2 + sum(b.values())/4
    zz = {key:value/4 for key,value in b.items()}
    for (i,j),value in b.items():
        h[i] -= value/4
        h[j] -= value/4
    bits = ((np.arange(2**n)[:,None] >> np.arange(n)) & 1).astype(int)
    z = 1 - 2*bits
    energy = constant + bits@a
    original = bits@np.array(cost,dtype=float)
    ising = offset + z@h
    for (i,j),value in b.items():
        energy += value*bits[:,i]*bits[:,j]
        ising += value*z[:,i]*z[:,j]/4
    for (i,j),value in pairs.items():
        original += value*bits[:,i]*bits[:,j]
    valid = np.array([feasible(x) for x in bits],dtype=bool)
    violation = np.zeros(2**n)
    for indices,target in equalities:
        violation += (bits[:,indices].sum(axis=1)-target)**2
    for i,j in forbidden:
        violation += bits[:,i]*bits[:,j]
    assert np.array_equal(valid,np.isclose(violation,0)), '원래 제약과 QUBO 제약을 대조하여 확인하세요.'
    assert np.allclose(energy,original+penalty*violation)
    assert valid.any() and np.allclose(energy,ising)
    assert np.allclose(energy[valid],original[valid])
    exact = original[valid].min()
    assert np.all(energy[~valid] > exact), '벌점 크기를 검토하세요.'
    optimum = valid & np.isclose(original,exact)
    scale = max(1.0,float(np.max(np.abs(energy))))

    def circuit(theta):
        qc = QuantumCircuit(n)
        qc.h(range(n))
        for layer in range(p):
            gamma = theta[2*layer]     # γ(감마): 비용 회로 각도
            beta = theta[2*layer+1]    # β(베타): 믹서 회로 각도
            for i,value in enumerate(h):
                qc.rz(2*gamma*value/scale,i)
            for (i,j),value in zz.items():
                qc.rzz(2*gamma*value/scale,i,j)
            for i in range(n):
                qc.rx(2*beta,i)
        return qc

    history = []
    def objective(theta):
        probability = Statevector.from_instruction(circuit(theta)).probabilities()
        value = float(probability@energy/scale)
        history.append(value)
        return value

    start = time.perf_counter()
    runs = []
    for gamma0,beta0 in [(0.8,0.3),(1.7,0.7)]:
        initial = np.array([gamma0,beta0]*p)
        runs.append(minimize(objective,initial,method='COBYLA',
                             options={'maxiter':maxiter,'rhobeg':0.5,'tol':1e-5}))
    result = min(runs,key=lambda r:r.fun)
    qc = circuit(result.x)
    state = Statevector.from_instruction(qc)
    probability = state.probabilities()
    state.seed(42)
    counts = state.sample_counts(shots)
    observed = np.array([counts.get(format(i,f'0{n}b'),0) for i in range(2**n)])
    candidates = np.flatnonzero(valid & (observed>0))
    print('큐비트 수:',n,'층수:',p,'측정 횟수:',shots)
    print('최적화 종료:',[(r.success,str(r.message)) for r in runs])
    print('총 목적함수 평가:',len(history),'실행시간(초):',round(time.perf_counter()-start,3))
    print('QUBO 계수:',constant,a.tolist(),b)
    print('Ising 계수:',offset,h.tolist(),zz)
    print('정규화 계수:',scale,'최적 각도:',result.x)
    print('정확 최적값:',exact)
    print('기대 QUBO 에너지:',float(probability@energy))
    print('유효해 확률:',float(probability[valid].sum()),'최적해 확률:',float(probability[optimum].sum()))
    print('유효해 관측 비율:',observed[valid].sum()/shots,'최적해 관측 비율:',observed[optimum].sum()/shots)
    if not len(candidates):
        print('유효한 해가 관측되지 않았습니다. 각도·층수·평가 횟수를 조정하세요.')
        return
    best = min(candidates,key=lambda i:(original[i],-observed[i]))
    mode = int(np.argmax(observed))
    print('최빈 비트열:',format(mode,f'0{n}b'),'제약 충족:',bool(valid[mode]))
    print('선택 비트(변수 순서):',bits[best].tolist())
    print('선택 항목:',[names[i] for i in range(n) if bits[best,i]])
    print('선택 목적값:',original[best],'정확해 차이:',original[best]-exact)
    describe(bits[best])
    print('목적함수 평가 이력(시작/종료):',history[0],history[-1])
    assert feasible(bits[best]) and original[best] >= exact-1e-8
    assert np.isclose(probability.sum(),1) and observed.sum()==shots
    print('회로 깊이:',qc.depth(),'게이트 수:',dict(qc.count_ops()))


jobs = ['A','B','C']
due = [1,2,3]
late_rate = [8,5,2]
setup = np.array([[0,2,7],[4,0,2],[3,6,0]])
names = [f'{j}_순서{t+1}' for j in jobs for t in range(3)]
cost = [max(0,t+1-due[j])*late_rate[j] for j in range(3) for t in range(3)]
pairs = {}
for t in range(2):
    for j in range(3):
        for k in range(3):
            if j != k:
                pair = tuple(sorted((3*j+t,3*k+t+1)))
                pairs[pair] = pairs.get(pair,0)+float(setup[j,k])
equalities = [([3*j+t for t in range(3)],1) for j in range(3)]
equalities += [([3*j+t for j in range(3)],1) for t in range(3)]
forbidden = []
def feasible(x):
    grid=x.reshape(3,3)
    return bool(np.all(grid.sum(axis=0)==1) and np.all(grid.sum(axis=1)==1))
def describe(x):
    grid=x.reshape(3,3)
    order=np.argmax(grid,axis=0)
    print('생산 순서:',[jobs[j] for j in order])
    print('작업별 1회 생산:',(grid.sum(axis=1)==1).tolist())
    print('순서별 1개 배정:',(grid.sum(axis=0)==1).tolist())
    print('지연비용/교체비용(만원):',np.dot(cost,x),sum(setup[order[t],order[t+1]] for t in range(2)))
solve(names,cost,pairs,equalities,forbidden,feasible,describe)

versions: {'qiskit': '2.2.3', 'numpy': '2.3.5', 'scipy': '1.16.3'}
큐비트 수: 9 층수: 2 측정 횟수: 4096
최적화 종료: [(False, 'Return from COBYLA because the objective function has been evaluated MAXFUN times.'), (False, 'Return from COBYLA because the objective function has been evaluated MAXFUN times.')]
총 목적함수 평가: 320 실행시간(초): 4.847
QUBO 계수: 600.0 [-200.0, -192.0, -184.0, -200.0, -200.0, -195.0, -200.0, -200.0, -200.0] {(0, 4): 2.0, (0, 7): 7.0, (1, 3): 4.0, (3, 7): 2.0, (1, 6): 3.0, (4, 6): 6.0, (1, 5): 2.0, (1, 8): 7.0, (2, 4): 4.0, (4, 8): 2.0, (2, 7): 3.0, (5, 7): 6.0, (0, 1): 200.0, (0, 2): 200.0, (1, 2): 200.0, (3, 4): 200.0, (3, 5): 200.0, (4, 5): 200.0, (6, 7): 200.0, (6, 8): 200.0, (7, 8): 200.0, (0, 3): 200.0, (0, 6): 200.0, (3, 6): 200.0, (1, 4): 200.0, (1, 7): 200.0, (4, 7): 200.0, (2, 5): 200.0, (2, 8): 200.0, (5, 8): 200.0}
Ising 계수: 626.5 [-102.25, -108.0, -109.75, -101.5, -103.5, -104.5, -102.25, -104.5, -102.25] {(0, 4): 0.5, (0, 7): 1.75, (1, 3): 1.0, (3, 7): 0.5, (1, 6): 0.75, (